# 실습 5. 재순위화(Reranking) — 다국어 Cross-Encoder 2단계 검색

## 실습 목표

1차 검색(Bi-Encoder)은 질문·문서를 **따로** 벡터화해 빠르지만 거칩니다. **Cross-Encoder** 는
(질문,문서)를 **함께** 보고 관련도를 직접 점수화해 상위 결과를 **정밀하게 재정렬**합니다(교안 5장).
이번엔 **정답 문서의 순위가 1차→리랭크에서 어떻게 오르는지** 봅니다.

- 1차 벡터(넓게, k=6) → 2차 Cross-Encoder(정밀, 재정렬)
- **한국어 정답을 끌어올리려면 다국어 리랭커**를 써야 한다 → `BAAI/bge-reranker-v2-m3` 사용
  (영어 위주 `ms-marco`는 한국어 정답을 오히려 밀어냄 — 교안 5.4)

> ⚠️ 최초 1회 `bge-reranker-v2-m3`(~2.3GB) 다운로드(HF 캐시). 한 번 받으면 재사용.

## 1. 준비 — 1차 벡터 검색기 + 다국어 리랭커

`rerank(query, docs, top_n, model=...)` 는 지정한 Cross-Encoder 로 후보를 재정렬합니다.
한국어 정답을 제대로 보려면 **다국어** `bge-reranker-v2-m3` 를 명시합니다(기본값 ms-marco는 영어 위주).

In [1]:
from _common import rag_tech_documents, build_vector_retriever, rerank, rank_of

RERANKER = "BAAI/bge-reranker-v2-m3"      # 다국어(한국어 양호) — 권장 모델
docs = rag_tech_documents()
vec = build_vector_retriever(docs, k=6, name="d21_rr_nb")   # 1차: 넓게 회수

def show(ds): return [f"{d.metadata['id']}/{d.metadata['topic']}" for d in ds]  # id+topic
def fmt(r):   return f"{r:>3}" if r is not None else "  ✗"

print("준비 완료 · 1차 벡터 k=6 · 리랭커:", RERANKER)

/Users/bkk/프로젝트/yeardream/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7120.63it/s]


준비 완료 · 1차 벡터 k=6 · 리랭커: BAAI/bge-reranker-v2-m3


In [3]:
print("docs : ",docs)
print("vec : ",vec)

docs :  [Document(metadata={'id': 'doc01', 'topic': '벡터검색'}, page_content='벡터 검색은 텍스트를 고차원 임베딩 벡터로 변환한 뒤 코사인 유사도 같은 지표로 가까운 문서를 찾는 검색 방식이다. 의미가 유사하면 단어가 달라도 검색되지만, 정확한 키워드 일치는 약하다.'), Document(metadata={'id': 'doc02', 'topic': 'BM25'}, page_content='BM25 는 단어 빈도와 문서 길이를 고려해 점수를 매기는 전통적인 키워드 검색 알고리즘이다. 정확한 단어 일치에 강하고 인덱싱이 빠르지만, 동의어나 의미 유사성은 잡지 못한다.'), Document(metadata={'id': 'doc03', 'topic': '하이브리드'}, page_content='하이브리드 검색은 BM25 같은 sparse 검색과 임베딩 기반 dense 검색을 결합해 각자의 약점을 보완한다. 두 결과의 점수를 가중 합산하거나 reciprocal rank fusion(RRF) 같은 방법으로 통합한다.'), Document(metadata={'id': 'doc04', 'topic': '리랭킹'}, page_content='Cross-Encoder 리랭커는 질의와 문서를 함께 입력 받아 관련성 점수를 직접 계산하는 모델이다. 1차 검색이 추려준 후보 20~100개를 정밀하게 재정렬해 Top-K 품질을 크게 끌어올린다. 비용이 비싸 1차 검색에는 쓰지 않는다.'), Document(metadata={'id': 'doc05', 'topic': 'Cohere리랭크'}, page_content='Cohere Rerank API 는 다국어를 지원하는 상용 리랭커 서비스다. rerank-multilingual-v3.0 모델은 한국어를 포함한 100개 이상 언어를 처리한다. API 호출 비용이 발생하므로 후보 수와 호출 빈도를 관리해야 한다.'), Document(metadata={'id': 'doc06', 'top

## 2. 정답(target) 기반 1차 vs 리랭크 순위 비교 — 순위가 오른다

각 질의는 **1차 벡터(Bi-Encoder)가 형제 개념에 헷갈려 정답을 1위로 못 올리는** 경우입니다.
Cross-Encoder(다국어 bge)는 (질문,문서)를 함께 보고 **정답을 1위로 끌어올립니다.**
(`id/topic` 을 함께 출력해 무엇이 바뀌는지 쉽게 확인)

In [4]:
SCENARIOS = [
    # 1차 벡터는 doc02(BM25)에 끌려 정답 doc03(하이브리드)을 2위로 → 리랭커가 1위로
    {"query": "BM25와 벡터를 합쳐 약점을 보완하는 검색", "target": "doc03"},
    # 1차 벡터는 doc04(리랭킹, '1차로 추린')에 끌려 정답 doc07(압축)을 2위로 → 리랭커가 1위로
    {"query": "1차로 추린 문서에서 핵심 문장만 추출", "target": "doc07"},
]

for s in SCENARIOS:
    q, tgt = s["query"], s["target"]
    cands = vec.invoke(q)                                       # 1차 후보(k=6)
    one = rank_of(tgt, cands)
    re_ranked = rerank(q, cands, top_n=6, model=RERANKER)       # 2차 재정렬(다국어)
    two = rank_of(tgt, re_ranked)
    print(f"🎯 {q!r}  (정답 {tgt})")
    print(f"   1차 벡터       : {fmt(one)}위   {show(cands)}")
    print(f"   bge 리랭크 후  : {fmt(two)}위   {show(re_ranked)}")
    print(f"   → 정답 {tgt} 순위 {one}위 → {two}위\n")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7140.25it/s]


🎯 'BM25와 벡터를 합쳐 약점을 보완하는 검색'  (정답 doc03)
   1차 벡터       :   2위   ['doc02/BM25', 'doc03/하이브리드', 'doc01/벡터검색', 'doc10/LangChain마이그레이션', 'doc06/쿼리변환', 'doc04/리랭킹']
   bge 리랭크 후  :   1위   ['doc03/하이브리드', 'doc02/BM25', 'doc01/벡터검색', 'doc06/쿼리변환', 'doc04/리랭킹', 'doc10/LangChain마이그레이션']
   → 정답 doc03 순위 2위 → 1위

🎯 '1차로 추린 문서에서 핵심 문장만 추출'  (정답 doc07)
   1차 벡터       :   2위   ['doc04/리랭킹', 'doc07/컨텍스트압축', 'doc10/LangChain마이그레이션', 'doc08/청킹', 'doc02/BM25', 'doc06/쿼리변환']
   bge 리랭크 후  :   1위   ['doc07/컨텍스트압축', 'doc04/리랭킹', 'doc10/LangChain마이그레이션', 'doc08/청킹', 'doc06/쿼리변환', 'doc02/BM25']
   → 정답 doc07 순위 2위 → 1위



**포인트 — '맞는 리랭커'를 고르면 정답이 올라간다**

- 1차 **Bi-Encoder(벡터)** 는 질문·문서를 따로 임베딩해 빠르지만, '형제 개념'(doc02 BM25, doc04 리랭킹)에
  끌려 **정답을 1위로 못 올림**(2위)
- 2차 **Cross-Encoder(bge, 다국어)** 는 (질문,문서)를 **함께** 보고 미세한 관련도를 판단 → **정답을 1위로 승격**
- 핵심: 리랭커도 **언어를 탄다** — 영어 위주 `ms-marco`는 한국어 정답을 오히려 밀어내므로(교안 5.4),
  한국어 서비스는 **다국어 `bge-reranker-v2-m3`**(또는 `Dongjin-kr/ko-reranker`, Cohere multilingual)를 써야 한다
- 즉 "리랭킹을 켰다"가 아니라 **"맞는(다국어) 리랭커를 골랐다"** 가 품질을 가른다

## 실습 정리

- **2단계 검색**(retrieve→rerank): 1차 Bi-Encoder로 넓게 회수 → 2차 Cross-Encoder로 정밀 재정렬
- 다국어 `bge-reranker-v2-m3` 는 Bi-Encoder가 놓친 한국어 정답을 **1위로 끌어올린다**(실측)
- 리랭커는 **언어를 타므로** 한국어엔 다국어 모델이 필수 — '맞는 리랭커 선택'이 품질을 좌우
- 다음 6교시: 하이브리드·멀티쿼리·리랭킹을 하나의 `pipeline()`으로 조립